In [3]:
# -*- coding: utf-8 -*-
"""
Screened ICE + 2D PDP interaction analysis for malodor descriptors

Purpose
-------
For reviewer comment:
"SHAP interpretation focuses on single functional groups, while landfill odor
is formed by co-existing multiple structural fragments. The paper needs to
supplement interaction analysis to reveal synergistic or antagonistic effects
between sulfur groups, amines, aldehydes and other typical substructures."

This script:
1. Analyzes only representative malodor descriptors.
2. Uses interpretable structure features only, not opaque Morgan bits.
3. Selects representative feature pairs from chemically meaningful groups.
4. Computes 2D PDP and PDP interaction residual.
5. Uses ICE curves as complementary conditional-response visualization.
6. Does not plot pairs without useful conclusions.

Interaction residual
--------------------
I(A, B) = f(A, B) - f(A) - f(B) + f0

I(A, B) > 0: model-level synergistic enhancement
I(A, B) < 0: model-level antagonistic / suppressive interaction
"""

import os
import re
import json
import warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

warnings.filterwarnings("ignore")


# ============================================================
# 0. Configuration
# ============================================================

RANDOM_SEED = 42

FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"
SHEET_NAME = 0

OUT_DIR = "./ICE_2D_PDP_representative_malodor_interactions"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
MODEL_DIR = os.path.join(OUT_DIR, "models")

for d in [OUT_DIR, PLOT_DIR, TABLE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

DPI = 600

TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

# 只分析恶臭/刺激性/填埋场相关描述词
MALODOR_LABELS = [
    "sulfurous",
    "garlic",
    "cabbage",
    "fishy",
    "pungent",
    "sharp",
    "sour",
    "cheesy",
    "sweaty",
    "musty"
]

# 每个恶臭标签最多画几组代表性交互图
MAX_PLOTS_PER_LABEL = 3

# 每个结构组进入候选池的 Top K 特征数
TOPK_FEATURES_PER_GROUP = 8

# 每个结构组组合最多评估多少个候选 pair
MAX_PAIRS_PER_GROUP_COMBO = 60

# PDP / ICE 样本量
MAX_SCREEN_PDP_SAMPLES = 700
MAX_FINAL_PDP_SAMPLES = 1500
MAX_ICE_SAMPLES = 300

# 连续变量网格点数
N_GRID_CONTINUOUS = 30

# 唯一值数量 <= 该值时，按离散变量处理
MAX_UNIQUE_AS_DISCRETE = 6

# 支持度阈值
MIN_POSITIVES_PER_LABEL = 10
MIN_FEATURE_SUPPORT_BINARY = 20
MIN_PAIR_COOC_SUPPORT_BINARY = 8

# 有解释价值的阈值，单位是预测概率
MIN_PDP_RANGE = 0.020
MIN_MEAN_ABS_RESIDUAL = 0.003
MIN_MAX_ABS_RESIDUAL = 0.010
MIN_BINARY_INTERACTION_CONTRAST = 0.020

# 二值特征冗余阈值
MAX_BINARY_PHI_CORR = 0.95
MAX_BINARY_JACCARD = 0.95

# ICE 图设置
MAX_ICE_LINES_PER_LEVEL = 80

# 是否优先人工指定的特征对；为空时完全自动
USE_MANUAL_PAIRS_FIRST = True


# ============================================================
# 1. XGBoost parameters
# ============================================================

BEST_PARAMS = {
    "n_estimators": 433,
    "max_depth": 7,
    "learning_rate": 0.0350057872293877,
    "subsample": 0.9947153135691092,
    "colsample_bytree": 0.7778835626400454,
    "min_child_weight": 1.0072775841844182,
    "reg_lambda": 3.4681854273849724,
    "reg_alpha": 6.955456414716767e-08,
    "gamma": 3.606069985094933
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)


# ============================================================
# 2. Structure groups
# ============================================================

GROUP_PATTERNS = {
    "Sulfur_groups": [
        r"sulfur", r"sulphur", r"groupscontainingsul", r"containing\s*sul",
        r"atom:\s*s", r"atomcount:\s*s", r"kg_element_s",
        r"thiol", r"thio", r"sulfide", r"sulfone", r"sulfoxide",
        r"sulfanyl", r"mercapto", r"disulfide", r"trisulfide",
        r"thiophene", r"\(-sh\)", r"n=c=s", r"isothiocyanate"
    ],

    "Amines": [
        r"amine", r"amines", r"amino", r"aniline", r"ammonia",
        r"ammonium", r"imino", r"imine", r"nitrogen",
        r"groupscontainingnitrogen", r"atom:\s*n", r"atomcount:\s*n",
        r"kg_element_n", r"pyridine", r"pyrrole", r"indole",
        r"n\s*≥", r"n\s*>=", r"n="
    ],

    "Aldehydes_Carbonyls": [
        r"aldehyde", r"aldehydic", r"formyl", r"carbonyl",
        r"c\(=o\)", r"\[cx3\]\(=o\)", r"\[cx3h1\]\(=o\)",
        r"cc\(=o\)", r"ketone", r"ketonic", r"acrolein",
        r"propanal", r"butanal", r"hexanal", r"benzaldehyde"
    ],

    "Carboxylic_acids": [
        r"carboxylic", r"carboxyl", r"-cooh", r"cooh",
        r"c\(=o\)\[ox2h1\]", r"c\(=o\)ox2h1",
        r"cc\(=o\)o", r"acid", r"fatty", r"acetic",
        r"propionic", r"butyric", r"valeric", r"isovaleric"
    ],

    "Ethers_Esters": [
        r"ether", r"ester", r"ethereal", r"acetal",
        r"lactone", r"acetate", r"c-o-c", r"coc"
    ],

    "Aromatics_Heterocycles": [
        r"aromatic", r"benzene", r"phenyl", r"toluene",
        r"xylene", r"styrene", r"heteroaromatic",
        r"thiophene", r"furan", r"pyridine", r"pyrrole",
        r"aromaticmonocyclic"
    ],
}


# 每个恶臭标签优先分析的结构组组合
LABEL_GROUP_PAIR_PLAN = {
    "sulfurous": [
        ("Sulfur_groups", "Amines"),
        ("Sulfur_groups", "Aldehydes_Carbonyls"),
        ("Sulfur_groups", "Ethers_Esters"),
    ],
    "garlic": [
        ("Sulfur_groups", "Amines"),
        ("Sulfur_groups", "Aldehydes_Carbonyls"),
        ("Sulfur_groups", "Ethers_Esters"),
    ],
    "cabbage": [
        ("Sulfur_groups", "Carboxylic_acids"),
        ("Sulfur_groups", "Amines"),
        ("Sulfur_groups", "Aldehydes_Carbonyls"),
    ],
    "fishy": [
        ("Amines", "Sulfur_groups"),
        ("Amines", "Aldehydes_Carbonyls"),
        ("Amines", "Ethers_Esters"),
    ],
    "pungent": [
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Aldehydes_Carbonyls", "Amines"),
        ("Sulfur_groups", "Amines"),
    ],
    "sharp": [
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Aldehydes_Carbonyls", "Amines"),
        ("Aldehydes_Carbonyls", "Ethers_Esters"),
    ],
    "sour": [
        ("Carboxylic_acids", "Sulfur_groups"),
        ("Carboxylic_acids", "Amines"),
        ("Carboxylic_acids", "Aldehydes_Carbonyls"),
    ],
    "cheesy": [
        ("Carboxylic_acids", "Sulfur_groups"),
        ("Carboxylic_acids", "Amines"),
        ("Carboxylic_acids", "Aldehydes_Carbonyls"),
    ],
    "sweaty": [
        ("Carboxylic_acids", "Sulfur_groups"),
        ("Carboxylic_acids", "Amines"),
        ("Carboxylic_acids", "Aldehydes_Carbonyls"),
    ],
    "musty": [
        ("Ethers_Esters", "Aromatics_Heterocycles"),
        ("Ethers_Esters", "Aldehydes_Carbonyls"),
        ("Aromatics_Heterocycles", "Sulfur_groups"),
    ],
}


# 可选：人工指定特征对。代码会先检查是否真实存在，不存在则跳过。
MANUAL_PAIRS = {
    "cheesy": [
        ("FG: Carboxylic acid(-COOH)", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: Carboxylic acid(-COOH)", "KG_Ancestor__Amines"),
        ("FG: Carboxylic acid(-COOH)", "KG_FG__Aldehyde"),
    ],
    "sour": [
        ("FG: Carboxylic acid(-COOH)", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: Carboxylic acid(-COOH)", "KG_Ancestor__Amines"),
        ("FG: Carboxylic acid(-COOH)", "KG_FG__Aldehyde"),
    ],
    "sulfurous": [
        ("FG: Thiol(-SH)", "KG_Ancestor__Amines"),
        ("FG: Thiol(-SH)", "KG_FG__Aldehyde"),
        ("KG_Ancestor__GroupsContainingSulfur", "KG_Ancestor__Amines"),
    ],
}


# ============================================================
# 3. General utilities
# ============================================================

def normalize_name(s):
    s = str(s)
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = s.replace("≥", ">=").replace("≤", "<=")
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "", s)
    return s.lower()


def resolve_feature_name(requested, columns):
    if requested in columns:
        return requested

    nr = normalize_name(requested)
    matches = [c for c in columns if normalize_name(c) == nr]

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        print(f"[WARN] Multiple matches for {requested}; use {matches[0]}")
        return matches[0]

    return None


def sanitize_filename(s):
    return re.sub(r"[^\w\-_\.]+", "_", str(s))


def find_smiles_col(df):
    candidates = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not candidates:
        return None

    priority = [
        "Canonical SMILES", "Canonical_SMILES", "canonical_smiles",
        "SMILES", "smiles", "StdSMILES"
    ]

    for p in priority:
        for c in candidates:
            if c.lower() == p.lower():
                return c

    return candidates[0]


def is_numeric_or_convertible(series):
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True

    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False


def is_interpretable_feature(feature):
    f = str(feature).lower()

    # 不用 Morgan bit 做交互解释图；它可用于建模，但不适合写结构机理。
    if f.startswith("morgan_"):
        return False

    # 过细 KG_REL / KG_RULE 一般不用于主图交互，除非你确实想解释。
    if f.startswith("kg_rel"):
        return False

    allowed_keywords = [
        "fg:", "kg_fg", "kg_ancestor", "kg_scaffold",
        "atom", "count", "exact", "molwt", "num", "c(=o)",
        "[cx3]", "n=c=s", "cooh", "thiol", "amine", "aldehyde",
        "sulf", "ether", "ester", "aromatic", "furan", "thiophene"
    ]

    return any(k in f for k in allowed_keywords)


def build_X_y(df):
    smiles_col = find_smiles_col(df)

    missing = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if missing:
        raise ValueError(f"Missing label columns: {missing}")

    y_df = df[TARGET_LABELS_24].fillna(0).astype(int)

    exclude = set(TARGET_LABELS_24)
    if smiles_col is not None:
        exclude.add(smiles_col)

    feature_cols = [c for c in df.columns if c not in exclude]

    good_cols = []
    bad_cols = []

    for c in feature_cols:
        if is_numeric_or_convertible(df[c]):
            good_cols.append(c)
        else:
            bad_cols.append(c)

    if bad_cols:
        print(f"[WARN] Dropped non-numeric feature columns: {len(bad_cols)}")
        print(bad_cols[:20])

    X_df = df[good_cols].copy()

    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X_df = X_df.fillna(0).astype(np.float32)

    return X_df, y_df, smiles_col


# ============================================================
# 4. Feature grouping
# ============================================================

def match_group(feature_name, group_name):
    f = str(feature_name).lower()
    return any(re.search(pat, f) for pat in GROUP_PATTERNS[group_name])


def assign_groups(feature_name):
    return [g for g in GROUP_PATTERNS if match_group(feature_name, g)]


def export_detected_group_features(feature_names):
    rows = []

    for i, f in enumerate(feature_names):
        groups = assign_groups(f)
        if groups and is_interpretable_feature(f):
            rows.append({
                "feature_index": i,
                "feature": f,
                "matched_groups": ";".join(groups)
            })

    out = pd.DataFrame(rows)

    out_path = os.path.join(TABLE_DIR, "detected_interpretable_group_features.csv")
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", out_path)

    return out


# ============================================================
# 5. Model
# ============================================================

def train_xgb_binary(X_values, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)

    model = xgb.XGBClassifier(**params)
    model.fit(X_values, y_bin)

    return model


def predict_proba(model, X_df):
    return model.predict_proba(X_df.values)[:, 1]


def get_xgb_gain_importance(model, feature_names):
    score = model.get_booster().get_score(importance_type="gain")
    importance = np.zeros(len(feature_names), dtype=float)

    for k, v in score.items():
        if k.startswith("f"):
            idx = int(k[1:])
            if 0 <= idx < len(feature_names):
                importance[idx] = float(v)

    return importance


# ============================================================
# 6. Support / redundancy checks
# ============================================================

def is_binary_feature(values):
    vals = pd.Series(values).dropna().unique()
    vals = set(float(v) for v in vals)
    return vals.issubset({0.0, 1.0})


def non_constant_feature(X_df, feature):
    return np.nanstd(X_df[feature].values.astype(float)) > 1e-12


def feature_support_ok(X_df, feature):
    if not non_constant_feature(X_df, feature):
        return False, "constant_feature"

    x = X_df[feature].values.astype(float)

    if is_binary_feature(x):
        n1 = int(np.sum(x > 0))
        n0 = int(np.sum(x <= 0))

        if n1 < MIN_FEATURE_SUPPORT_BINARY:
            return False, f"binary_positive_support_too_low_n1={n1}"

        if n0 < MIN_FEATURE_SUPPORT_BINARY:
            return False, f"binary_negative_support_too_low_n0={n0}"

    return True, "ok"


def pair_support_ok(X_df, feature_a, feature_b):
    xa = X_df[feature_a].values.astype(float)
    xb = X_df[feature_b].values.astype(float)

    support = {
        "n00": np.nan,
        "n10": np.nan,
        "n01": np.nan,
        "n11": np.nan,
        "pair_support_status": "not_binary_pair"
    }

    if is_binary_feature(xa) and is_binary_feature(xb):
        a = (xa > 0).astype(int)
        b = (xb > 0).astype(int)

        n00 = int(np.sum((a == 0) & (b == 0)))
        n10 = int(np.sum((a == 1) & (b == 0)))
        n01 = int(np.sum((a == 0) & (b == 1)))
        n11 = int(np.sum((a == 1) & (b == 1)))

        support.update({
            "n00": n00,
            "n10": n10,
            "n01": n01,
            "n11": n11,
            "pair_support_status": "binary_pair"
        })

        if n11 < MIN_PAIR_COOC_SUPPORT_BINARY:
            return False, f"cooccurrence_too_low_n11={n11}", support

    return True, "ok", support


def binary_redundancy_stats(X_df, feature_a, feature_b):
    xa = X_df[feature_a].values.astype(float)
    xb = X_df[feature_b].values.astype(float)

    out = {
        "binary_phi_corr": np.nan,
        "binary_jaccard": np.nan,
        "redundant_pair": False,
        "redundancy_reason": "ok"
    }

    if not (is_binary_feature(xa) and is_binary_feature(xb)):
        return out

    a = (xa > 0).astype(int)
    b = (xb > 0).astype(int)

    if np.std(a) == 0 or np.std(b) == 0:
        out["redundant_pair"] = True
        out["redundancy_reason"] = "binary_constant"
        return out

    phi = float(np.corrcoef(a, b)[0, 1])

    intersection = np.sum((a == 1) & (b == 1))
    union = np.sum((a == 1) | (b == 1))
    jaccard = float(intersection / union) if union > 0 else np.nan

    out["binary_phi_corr"] = phi
    out["binary_jaccard"] = jaccard

    if np.isfinite(phi) and abs(phi) >= MAX_BINARY_PHI_CORR:
        out["redundant_pair"] = True
        out["redundancy_reason"] = f"high_phi_corr={phi:.3f}"

    if np.isfinite(jaccard) and jaccard >= MAX_BINARY_JACCARD:
        out["redundant_pair"] = True
        out["redundancy_reason"] = f"high_jaccard={jaccard:.3f}"

    return out


# ============================================================
# 7. PDP / ICE computation
# ============================================================

def sample_X(X_df, max_n, random_seed=42):
    if len(X_df) <= max_n:
        return X_df.copy()

    rng = np.random.default_rng(random_seed)
    idx = rng.choice(np.arange(len(X_df)), size=max_n, replace=False)
    return X_df.iloc[idx].copy()


def make_grid(values, n_grid=N_GRID_CONTINUOUS, max_unique_as_discrete=MAX_UNIQUE_AS_DISCRETE):
    values = pd.Series(values).dropna().astype(float)
    unique_vals = np.sort(values.unique())

    if len(unique_vals) <= max_unique_as_discrete:
        return unique_vals

    qs = np.linspace(0.05, 0.95, n_grid)
    grid = np.quantile(values, qs)
    grid = np.unique(np.round(grid, 6))

    return grid


def compute_1d_pdp(model, X_ref, feature, grid):
    vals = []

    for v in grid:
        X_tmp = X_ref.copy()
        X_tmp[feature] = v
        vals.append(float(np.mean(predict_proba(model, X_tmp))))

    return np.array(vals)


def compute_2d_pdp(model, X_ref, feature_a, grid_a, feature_b, grid_b):
    mat = np.zeros((len(grid_b), len(grid_a)), dtype=float)

    for i, vb in enumerate(grid_b):
        for j, va in enumerate(grid_a):
            X_tmp = X_ref.copy()
            X_tmp[feature_a] = va
            X_tmp[feature_b] = vb
            mat[i, j] = float(np.mean(predict_proba(model, X_tmp)))

    return mat


def compute_interaction_residual(pdp2d, pdp_a, pdp_b, baseline):
    residual = np.zeros_like(pdp2d)

    for i in range(len(pdp_b)):
        for j in range(len(pdp_a)):
            residual[i, j] = pdp2d[i, j] - pdp_a[j] - pdp_b[i] + baseline

    return residual


def compute_ice_curves(model, X_ref, feature_a, grid_a, feature_b, b_levels):
    out = {}

    for vb in b_levels:
        curves = []

        for va in grid_a:
            X_tmp = X_ref.copy()
            X_tmp[feature_a] = va
            X_tmp[feature_b] = vb
            curves.append(predict_proba(model, X_tmp))

        out[vb] = np.vstack(curves).T

    return out


# ============================================================
# 8. Interaction judgment
# ============================================================

def judge_interaction(metrics):
    if metrics.get("redundant_pair", False):
        return False, "redundant_features", metrics.get("redundancy_reason", "redundant_pair")

    pdp_range = metrics["pdp_range"]
    mean_abs = metrics["mean_abs_interaction_residual"]
    max_abs = metrics["max_abs_interaction_residual"]
    contrast = metrics["interaction_contrast_2x2"]

    reasons = []

    if pdp_range < MIN_PDP_RANGE:
        reasons.append(f"pdp_range_too_small={pdp_range:.4f}")

    residual_signal = (mean_abs >= MIN_MEAN_ABS_RESIDUAL) or (max_abs >= MIN_MAX_ABS_RESIDUAL)
    contrast_signal = np.isfinite(contrast) and abs(contrast) >= MIN_BINARY_INTERACTION_CONTRAST

    if not residual_signal and not contrast_signal:
        reasons.append(
            f"interaction_too_weak: mean_abs={mean_abs:.4f}, max_abs={max_abs:.4f}, contrast={contrast:.4f}"
        )

    if reasons:
        return False, "no_clear_interaction", "; ".join(reasons)

    signal = contrast if np.isfinite(contrast) else metrics["signed_mean_interaction_residual"]

    if signal > 0:
        return True, "synergistic_positive", "positive interaction signal"
    elif signal < 0:
        return True, "antagonistic_negative", "negative interaction signal"
    else:
        return True, "interaction_strength_only", "non-directional interaction strength"


def compute_pair_result(model, X_df, label, feature_a, feature_b, max_samples):
    ok_a, reason_a = feature_support_ok(X_df, feature_a)
    if not ok_a:
        return None, f"feature_a_failed: {reason_a}"

    ok_b, reason_b = feature_support_ok(X_df, feature_b)
    if not ok_b:
        return None, f"feature_b_failed: {reason_b}"

    ok_pair, pair_reason, support = pair_support_ok(X_df, feature_a, feature_b)
    if not ok_pair:
        return None, f"pair_failed: {pair_reason}"

    redundancy = binary_redundancy_stats(X_df, feature_a, feature_b)

    X_ref = sample_X(X_df, max_samples, RANDOM_SEED)

    grid_a = make_grid(X_ref[feature_a].values)
    grid_b = make_grid(X_ref[feature_b].values)

    if len(grid_a) < 2 or len(grid_b) < 2:
        return None, "grid_too_small"

    baseline = float(np.mean(predict_proba(model, X_ref)))

    pdp_a = compute_1d_pdp(model, X_ref, feature_a, grid_a)
    pdp_b = compute_1d_pdp(model, X_ref, feature_b, grid_b)
    pdp2d = compute_2d_pdp(model, X_ref, feature_a, grid_a, feature_b, grid_b)
    residual = compute_interaction_residual(pdp2d, pdp_a, pdp_b, baseline)

    interaction_contrast = np.nan
    interaction_direction = "not_2x2"

    if len(grid_a) == 2 and len(grid_b) == 2:
        # matrix: rows = B, columns = A
        # assumes grids contain 0 and 1
        try:
            ia0 = int(np.where(grid_a == 0)[0][0])
            ia1 = int(np.where(grid_a == 1)[0][0])
            ib0 = int(np.where(grid_b == 0)[0][0])
            ib1 = int(np.where(grid_b == 1)[0][0])

            f00 = float(pdp2d[ib0, ia0])
            f10 = float(pdp2d[ib0, ia1])
            f01 = float(pdp2d[ib1, ia0])
            f11 = float(pdp2d[ib1, ia1])

            interaction_contrast = f11 - f10 - f01 + f00

            if interaction_contrast > 0:
                interaction_direction = "synergistic_positive"
            elif interaction_contrast < 0:
                interaction_direction = "antagonistic_negative"
            else:
                interaction_direction = "near_zero"
        except Exception:
            pass

    metrics = {
        "label": label,
        "feature_a": feature_a,
        "feature_b": feature_b,
        "baseline_probability": baseline,
        "grid_a_size": len(grid_a),
        "grid_b_size": len(grid_b),
        "grid_a": ";".join(map(str, grid_a)),
        "grid_b": ";".join(map(str, grid_b)),
        "min_PDP2D": float(np.min(pdp2d)),
        "max_PDP2D": float(np.max(pdp2d)),
        "pdp_range": float(np.max(pdp2d) - np.min(pdp2d)),
        "min_interaction_residual": float(np.min(residual)),
        "max_interaction_residual": float(np.max(residual)),
        "mean_abs_interaction_residual": float(np.mean(np.abs(residual))),
        "max_abs_interaction_residual": float(np.max(np.abs(residual))),
        "signed_mean_interaction_residual": float(np.mean(residual)),
        "interaction_contrast_2x2": float(interaction_contrast) if np.isfinite(interaction_contrast) else np.nan,
        "interaction_direction_2x2": interaction_direction,
    }

    metrics.update(support)
    metrics.update(redundancy)

    useful, conclusion, reason = judge_interaction(metrics)
    metrics["useful_for_plot"] = useful
    metrics["conclusion_type"] = conclusion
    metrics["decision_reason"] = reason

    return {
        "metrics": metrics,
        "grid_a": grid_a,
        "grid_b": grid_b,
        "pdp2d": pdp2d,
        "residual": residual,
    }, "ok"


# ============================================================
# 9. Candidate feature and pair selection
# ============================================================

def candidate_features_by_group(group, X_df, feature_names, importance):
    rows = []

    for idx, f in enumerate(feature_names):
        if not is_interpretable_feature(f):
            continue

        if not match_group(f, group):
            continue

        ok, reason = feature_support_ok(X_df, f)
        if not ok:
            continue

        rows.append({
            "group": group,
            "feature_index": idx,
            "feature": f,
            "gain_importance": float(importance[idx]),
            "support_reason": reason,
            "matched_groups": ";".join(assign_groups(f)),
        })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values("gain_importance", ascending=False).head(TOPK_FEATURES_PER_GROUP)


def get_manual_candidate_pairs(label, X_df, feature_names, importance):
    rows = []

    if not USE_MANUAL_PAIRS_FIRST or label not in MANUAL_PAIRS:
        return rows

    for fa_raw, fb_raw in MANUAL_PAIRS[label]:
        fa = resolve_feature_name(fa_raw, X_df.columns)
        fb = resolve_feature_name(fb_raw, X_df.columns)

        if fa is None or fb is None:
            continue

        ia = feature_names.index(fa)
        ib = feature_names.index(fb)

        rows.append({
            "label": label,
            "group_a": assign_groups(fa)[0] if assign_groups(fa) else "Manual",
            "feature_a": fa,
            "importance_a": float(importance[ia]),
            "group_b": assign_groups(fb)[0] if assign_groups(fb) else "Manual",
            "feature_b": fb,
            "importance_b": float(importance[ib]),
            "selection_mode": "manual_curated"
        })

    return rows


def screen_candidate_pairs(label, model, X_df, feature_names, importance):
    rows = []
    used_pairs = set()

    # 1. Manual candidates
    for r in get_manual_candidate_pairs(label, X_df, feature_names, importance):
        key = tuple(sorted([r["feature_a"], r["feature_b"]]))
        if key not in used_pairs:
            rows.append(r)
            used_pairs.add(key)

    # 2. Auto candidates from chemical group plan
    plan = LABEL_GROUP_PAIR_PLAN.get(label, [])

    for group_a, group_b in plan:
        cand_a = candidate_features_by_group(group_a, X_df, feature_names, importance)
        cand_b = candidate_features_by_group(group_b, X_df, feature_names, importance)

        if cand_a.empty or cand_b.empty:
            continue

        count = 0

        for _, ra in cand_a.iterrows():
            for _, rb in cand_b.iterrows():
                fa = ra["feature"]
                fb = rb["feature"]

                if fa == fb:
                    continue

                key = tuple(sorted([fa, fb]))
                if key in used_pairs:
                    continue

                rows.append({
                    "label": label,
                    "group_a": group_a,
                    "feature_a": fa,
                    "importance_a": float(ra["gain_importance"]),
                    "group_b": group_b,
                    "feature_b": fb,
                    "importance_b": float(rb["gain_importance"]),
                    "selection_mode": "auto_group_importance"
                })

                used_pairs.add(key)
                count += 1

                if count >= MAX_PAIRS_PER_GROUP_COMBO:
                    break

            if count >= MAX_PAIRS_PER_GROUP_COMBO:
                break

    if not rows:
        return pd.DataFrame()

    # 3. Compute quick PDP metrics for all candidate pairs
    evaluated = []

    for r in rows:
        result, status = compute_pair_result(
            model=model,
            X_df=X_df,
            label=label,
            feature_a=r["feature_a"],
            feature_b=r["feature_b"],
            max_samples=MAX_SCREEN_PDP_SAMPLES
        )

        base = dict(r)
        base["screen_status"] = status

        if result is None:
            base["useful_for_plot"] = False
            base["conclusion_type"] = "failed_before_metrics"
            base["decision_reason"] = status
            evaluated.append(base)
            continue

        base.update(result["metrics"])
        evaluated.append(base)

    eval_df = pd.DataFrame(evaluated)

    all_path = os.path.join(TABLE_DIR, f"ALL_candidate_pair_decision__{label}.csv")
    eval_df.to_csv(all_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", all_path)

    useful_df = eval_df[eval_df["useful_for_plot"] == True].copy()

    if useful_df.empty:
        return useful_df

    useful_df = useful_df.sort_values(
        ["mean_abs_interaction_residual", "max_abs_interaction_residual", "pdp_range"],
        ascending=False
    ).reset_index(drop=True)

    # Keep diversity across group combinations
    selected = []
    used_group_combos = set()

    for _, row in useful_df.iterrows():
        combo = tuple(sorted([row["group_a"], row["group_b"]]))

        if combo in used_group_combos:
            continue

        selected.append(row)
        used_group_combos.add(combo)

        if len(selected) >= MAX_PLOTS_PER_LABEL:
            break

    if len(selected) < MAX_PLOTS_PER_LABEL:
        selected_keys = set(tuple(sorted([r["feature_a"], r["feature_b"]])) for r in selected)

        for _, row in useful_df.iterrows():
            key = tuple(sorted([row["feature_a"], row["feature_b"]]))

            if key in selected_keys:
                continue

            selected.append(row)
            selected_keys.add(key)

            if len(selected) >= MAX_PLOTS_PER_LABEL:
                break

    selected_df = pd.DataFrame(selected).reset_index(drop=True)
    selected_df.insert(1, "plot_pair_id", np.arange(1, len(selected_df) + 1))

    selected_path = os.path.join(TABLE_DIR, f"USEFUL_pairs_selected_for_plot__{label}.csv")
    selected_df.to_csv(selected_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", selected_path)

    return selected_df


# ============================================================
# 10. Plotting: unified blue-white style with Arial bold font
# ============================================================

import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import font_manager as fm


# ------------------------------------------------------------
# 10.1 Global publication-style figure settings
# ------------------------------------------------------------

FONT_TITLE = 16
FONT_LABEL = 16
FONT_TICK = 16
FONT_CBAR = 16
FONT_ANNOT = 16
FONT_LEGEND = 16

AXIS_LINEWIDTH = 2.0
TICK_WIDTH = 2.0
TICK_LENGTH = 6
MEAN_LINE_WIDTH = 3.6
ICE_LINE_WIDTH = 1.0

# 统一蓝白色系
BLUE_WHITE_CMAP = LinearSegmentedColormap.from_list(
    "custom_blue_white",
    [
        "#ffffff",
        "#eff6fb",
        "#d9eaf7",
        "#bdd7e7",
        "#6baed6",
        "#3182bd",
        "#08519c",
        "#08306b"
    ]
)

ICE_MEAN_COLORS = [
    "#08306b",  # dark blue
    "#2171b5",
    "#6baed6",
    "#9ecae1"
]

ICE_INDIVIDUAL_COLOR = "#9ecae1"


def setup_matplotlib_style():
    available_fonts = {f.name for f in fm.fontManager.ttflist}

    if "Arial" in available_fonts:
        font_family = "Arial"
    else:
        font_family = "DejaVu Sans"
        print("[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.")

    mpl.rcParams.update({
        "font.family": font_family,
        "font.weight": "bold",
        "axes.titlesize": FONT_TITLE,
        "axes.titleweight": "bold",
        "axes.labelsize": FONT_LABEL,
        "axes.labelweight": "bold",
        "xtick.labelsize": FONT_TICK,
        "ytick.labelsize": FONT_TICK,
        "legend.fontsize": FONT_LEGEND,
        "axes.linewidth": AXIS_LINEWIDTH,
        "xtick.major.width": TICK_WIDTH,
        "ytick.major.width": TICK_WIDTH,
        "xtick.major.size": TICK_LENGTH,
        "ytick.major.size": TICK_LENGTH,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.unicode_minus": False,
    })


setup_matplotlib_style()


def style_axis(ax):
    ax.tick_params(
        axis="both",
        which="major",
        width=TICK_WIDTH,
        length=TICK_LENGTH,
        labelsize=FONT_TICK
    )

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(AXIS_LINEWIDTH)


def style_colorbar(cbar, label):
    cbar.set_label(label, fontsize=FONT_CBAR, fontweight="bold")
    cbar.ax.tick_params(labelsize=FONT_TICK, width=TICK_WIDTH, length=TICK_LENGTH)

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")


def is_discrete_grid(grid):
    return len(grid) <= MAX_UNIQUE_AS_DISCRETE


def format_grid_labels(grid):
    labels = []

    for v in grid:
        try:
            vf = float(v)
            if vf.is_integer():
                labels.append(str(int(vf)))
            else:
                labels.append(f"{vf:.3g}")
        except Exception:
            labels.append(str(v))

    return labels


# ------------------------------------------------------------
# 10.2 Discrete / binary PDP block heatmap
# ------------------------------------------------------------

def plot_discrete_heatmap(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    title,
    cbar_label,
    out_png,
    signed=False
):
    """
    适用于二值或低水平离散结构特征。

    signed=False:
        颜色表示 PDP predicted probability。

    signed=True:
        颜色统一用蓝白色表示 |interaction residual| 的强度；
        单元格文字保留正负号，用于判断协同/拮抗方向。
    """

    mat = np.asarray(matrix, dtype=float)

    if signed:
        color_mat = np.abs(mat)
        vmin = 0.0
        vmax = float(np.nanmax(color_mat))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        colorbar_label = f"|{cbar_label}|"
    else:
        color_mat = mat
        vmin = float(np.nanmin(color_mat))
        vmax = float(np.nanmax(color_mat))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0
        colorbar_label = cbar_label

    fig_w = max(6.6, 1.25 * len(grid_a))
    fig_h = max(5.4, 1.10 * len(grid_b))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        color_mat,
        origin="lower",
        aspect="auto",
        cmap=BLUE_WHITE_CMAP,
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(im, ax=ax)
    style_colorbar(cbar, colorbar_label)

    ax.set_xticks(np.arange(len(grid_a)))
    ax.set_yticks(np.arange(len(grid_b)))

    ax.set_xticklabels(format_grid_labels(grid_a), fontweight="bold")
    ax.set_yticklabels(format_grid_labels(grid_b), fontweight="bold")

    ax.set_xlabel(feature_a, fontsize=FONT_LABEL, fontweight="bold")
    ax.set_ylabel(feature_b, fontsize=FONT_LABEL, fontweight="bold")
    ax.set_title(title, fontsize=FONT_TITLE, fontweight="bold", pad=18)

    # 数值标注
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            raw_val = mat[i, j]
            color_val = color_mat[i, j]

            if not np.isfinite(raw_val):
                continue

            if signed:
                text = f"{raw_val:+.3f}"
            else:
                text = f"{raw_val:.3f}"

            text_color = "white" if color_val > (vmin + 0.62 * (vmax - vmin)) else "black"

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=FONT_ANNOT,
                fontweight="bold",
                color=text_color
            )

    # 白色网格线强调离散组合
    ax.set_xticks(np.arange(-0.5, len(grid_a), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(grid_b), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=2.0)
    ax.tick_params(which="minor", bottom=False, left=False)

    style_axis(ax)

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    print("[SAVE]", out_png)


# ------------------------------------------------------------
# 10.3 Continuous 2D PDP contour plot
# ------------------------------------------------------------

def plot_contour(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    title,
    cbar_label,
    out_png,
    signed=False
):
    """
    适用于连续或多水平数值特征。

    signed=False:
        颜色表示 PDP predicted probability。

    signed=True:
        颜色统一用蓝白色表示 |interaction residual|；
        等高线仍标注原 residual 数值，用于判断正负方向。
    """

    Xg, Yg = np.meshgrid(grid_a, grid_b)
    mat = np.asarray(matrix, dtype=float)

    if signed:
        color_mat = np.abs(mat)
        vmin = 0.0
        vmax = float(np.nanmax(color_mat))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        levels = np.linspace(vmin, vmax, 24)
        colorbar_label = f"|{cbar_label}|"
    else:
        color_mat = mat
        vmin = float(np.nanmin(color_mat))
        vmax = float(np.nanmax(color_mat))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0
        levels = np.linspace(vmin, vmax, 24)
        colorbar_label = cbar_label

    fig, ax = plt.subplots(figsize=(7.8, 6.3))

    cf = ax.contourf(
        Xg,
        Yg,
        color_mat,
        levels=levels,
        cmap=BLUE_WHITE_CMAP,
        extend="both"
    )

    cbar = plt.colorbar(cf, ax=ax)
    style_colorbar(cbar, colorbar_label)

    # 等高线统一深蓝色
    try:
        cs = ax.contour(
            Xg,
            Yg,
            mat,
            levels=8,
            colors="#08306b",
            linewidths=1.3,
            alpha=0.85
        )

        ax.clabel(
            cs,
            inline=True,
            fontsize=FONT_ANNOT - 4,
            fmt="%.3f",
            colors="#08306b"
        )
    except Exception:
        pass

    ax.set_xlabel(feature_a, fontsize=FONT_LABEL, fontweight="bold")
    ax.set_ylabel(feature_b, fontsize=FONT_LABEL, fontweight="bold")
    ax.set_title(title, fontsize=FONT_TITLE, fontweight="bold", pad=18)

    style_axis(ax)

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    print("[SAVE]", out_png)


# ------------------------------------------------------------
# 10.4 Auto-select PDP plotting style
# ------------------------------------------------------------

def plot_pdp2d_auto(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    title,
    cbar_label,
    out_png,
    signed=False
):
    """
    自动选择 PDP 图类型：

    1. 任一轴为二值或低水平离散变量：
       使用 block heatmap，避免连续插值造成误导。

    2. 两个轴均为连续或多水平数值变量：
       使用 2D contour PDP。
    """

    discrete_a = is_discrete_grid(grid_a)
    discrete_b = is_discrete_grid(grid_b)

    if discrete_a or discrete_b:
        plot_discrete_heatmap(
            matrix=matrix,
            grid_a=grid_a,
            grid_b=grid_b,
            feature_a=feature_a,
            feature_b=feature_b,
            title=title,
            cbar_label=cbar_label,
            out_png=out_png,
            signed=signed
        )
    else:
        plot_contour(
            matrix=matrix,
            grid_a=grid_a,
            grid_b=grid_b,
            feature_a=feature_a,
            feature_b=feature_b,
            title=title,
            cbar_label=cbar_label,
            out_png=out_png,
            signed=signed
        )


# ------------------------------------------------------------
# 10.5 ICE curves with matched font style
# ------------------------------------------------------------

def plot_ice_curves(
    model,
    X_df,
    label,
    feature_a,
    feature_b,
    grid_a,
    grid_b,
    out_png
):
    """
    ICE 曲线作为补充图。
    字体和线条风格与 PDP 图保持一致。
    """

    X_ice = sample_X(X_df, MAX_ICE_SAMPLES, RANDOM_SEED)

    if len(grid_b) > 3:
        b_levels = [grid_b[0], grid_b[len(grid_b) // 2], grid_b[-1]]
    else:
        b_levels = list(grid_b)

    ice = compute_ice_curves(
        model=model,
        X_ref=X_ice,
        feature_a=feature_a,
        grid_a=grid_a,
        feature_b=feature_b,
        b_levels=b_levels
    )

    fig, ax = plt.subplots(figsize=(7.8, 6.0))
    rng = np.random.default_rng(RANDOM_SEED)

    for idx_level, (vb, curves) in enumerate(ice.items()):
        n = curves.shape[0]

        if n > MAX_ICE_LINES_PER_LEVEL:
            idx = rng.choice(np.arange(n), size=MAX_ICE_LINES_PER_LEVEL, replace=False)
        else:
            idx = np.arange(n)

        # 个体曲线使用浅蓝色
        for k in idx:
            ax.plot(
                grid_a,
                curves[k],
                color=ICE_INDIVIDUAL_COLOR,
                alpha=0.16,
                linewidth=ICE_LINE_WIDTH
            )

        mean_curve = curves.mean(axis=0)
        mean_color = ICE_MEAN_COLORS[idx_level % len(ICE_MEAN_COLORS)]

        ax.plot(
            grid_a,
            mean_curve,
            color=mean_color,
            linewidth=MEAN_LINE_WIDTH,
            marker="o",
            markersize=8,
            markeredgewidth=1.5,
            label=f"{feature_b}={vb}"
        )

    ax.set_xlabel(feature_a, fontsize=FONT_LABEL, fontweight="bold")
    ax.set_ylabel(f"Predicted probability of {label}", fontsize=FONT_LABEL, fontweight="bold")
    ax.set_title(
        f"{label}: ICE curves of {feature_a}\nconditioned on {feature_b}",
        fontsize=FONT_TITLE,
        fontweight="bold",
        pad=18
    )

    legend = ax.legend(frameon=False, fontsize=FONT_LEGEND)

    for text in legend.get_texts():
        text.set_fontweight("bold")

    style_axis(ax)

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    print("[SAVE]", out_png)


# ------------------------------------------------------------
# 10.6 Save PDP matrices
# ------------------------------------------------------------

def save_pair_matrices(
    label,
    pair_id,
    feature_a,
    feature_b,
    grid_a,
    grid_b,
    pdp2d,
    residual
):
    prefix = (
        f"{label}__pair{pair_id:02d}__"
        f"{sanitize_filename(feature_a)}__x__{sanitize_filename(feature_b)}"
    )

    pd.DataFrame(
        pdp2d,
        index=[f"{feature_b}={v}" for v in grid_b],
        columns=[f"{feature_a}={v}" for v in grid_a]
    ).to_csv(
        os.path.join(TABLE_DIR, f"PDP2D_probability_matrix__{prefix}.csv"),
        encoding="utf-8-sig"
    )

    pd.DataFrame(
        residual,
        index=[f"{feature_b}={v}" for v in grid_b],
        columns=[f"{feature_a}={v}" for v in grid_a]
    ).to_csv(
        os.path.join(TABLE_DIR, f"PDP2D_interaction_residual_matrix__{prefix}.csv"),
        encoding="utf-8-sig"
    )


# ============================================================
# 11. 保存矩阵 + 最终特征对绘图函数
# ============================================================

def save_pair_matrices(
    label,
    pair_id,
    feature_a,
    feature_b,
    grid_a,
    grid_b,
    pdp2d,
    residual
):
    """
    保存每个特征对的 2D PDP 概率矩阵和 PDP interaction residual 矩阵。
    无论最终是否画图，矩阵都会保存，方便后续检查。
    """

    prefix = (
        f"{label}__pair{pair_id:02d}__"
        f"{sanitize_filename(feature_a)}__x__{sanitize_filename(feature_b)}"
    )

    pd.DataFrame(
        pdp2d,
        index=[f"{feature_b}={v}" for v in grid_b],
        columns=[f"{feature_a}={v}" for v in grid_a]
    ).to_csv(
        os.path.join(TABLE_DIR, f"PDP2D_probability_matrix__{prefix}.csv"),
        encoding="utf-8-sig"
    )

    pd.DataFrame(
        residual,
        index=[f"{feature_b}={v}" for v in grid_b],
        columns=[f"{feature_a}={v}" for v in grid_a]
    ).to_csv(
        os.path.join(TABLE_DIR, f"PDP2D_interaction_residual_matrix__{prefix}.csv"),
        encoding="utf-8-sig"
    )


def plot_final_pair(model, X_df, label, pair_id, pair_row):
    """
    对筛选后的有用特征对进行最终计算和绘图。

    输出：
    1. PDP2D probability matrix
    2. PDP2D interaction residual matrix
    3. PDP2D probability 图
    4. PDP2D interaction residual 图
    5. ICE curves 图

    注意：
    - 如果 final 重新计算后 useful_for_plot=False，则只保存矩阵，不画图。
    - 离散/二值特征自动画 block heatmap。
    - 连续特征自动画 contour plot。
    """

    feature_a = pair_row["feature_a"]
    feature_b = pair_row["feature_b"]

    result, status = compute_pair_result(
        model=model,
        X_df=X_df,
        label=label,
        feature_a=feature_a,
        feature_b=feature_b,
        max_samples=MAX_FINAL_PDP_SAMPLES
    )

    if result is None:
        print(f"[SKIP FINAL] {label}: {feature_a} × {feature_b}: {status}")
        return None

    metrics = result["metrics"]
    grid_a = result["grid_a"]
    grid_b = result["grid_b"]
    pdp2d = result["pdp2d"]
    residual = result["residual"]

    # 先保存矩阵
    save_pair_matrices(
        label=label,
        pair_id=pair_id,
        feature_a=feature_a,
        feature_b=feature_b,
        grid_a=grid_a,
        grid_b=grid_b,
        pdp2d=pdp2d,
        residual=residual
    )

    # 如果最终重新计算后判断没有有用结论，则不画图
    if not metrics["useful_for_plot"]:
        print(
            f"[NO PLOT FINAL] {label}: {feature_a} × {feature_b}: "
            f"{metrics['decision_reason']}"
        )
        return metrics

    prefix = (
        f"{label}__pair{pair_id:02d}__"
        f"{sanitize_filename(feature_a)}__x__{sanitize_filename(feature_b)}"
    )

    # 1. 2D PDP probability 图
    plot_pdp2d_auto(
        matrix=pdp2d,
        grid_a=grid_a,
        grid_b=grid_b,
        feature_a=feature_a,
        feature_b=feature_b,
        title=f"{label}: 2D PDP interaction\n{metrics['conclusion_type']}",
        cbar_label="Partial dependence / predicted probability",
        out_png=os.path.join(PLOT_DIR, f"PDP2D_probability__{prefix}.png"),
        signed=False
    )

    # 2. PDP interaction residual 图
    plot_pdp2d_auto(
        matrix=residual,
        grid_a=grid_a,
        grid_b=grid_b,
        feature_a=feature_a,
        feature_b=feature_b,
        title=f"{label}: PDP interaction residual\nI(A,B)=f(A,B)-f(A)-f(B)+baseline",
        cbar_label="PDP interaction residual",
        out_png=os.path.join(PLOT_DIR, f"PDP2D_interaction_residual__{prefix}.png"),
        signed=True
    )

    # 3. ICE curves 图
    plot_ice_curves(
        model=model,
        X_df=X_df,
        label=label,
        feature_a=feature_a,
        feature_b=feature_b,
        grid_a=grid_a,
        grid_b=grid_b,
        out_png=os.path.join(PLOT_DIR, f"ICE_curves__{prefix}.png")
    )

    return metrics


# ============================================================
# 11. Per-label workflow
# ============================================================

def analyze_one_label(label, X_df, y_df, feature_names):
    print("\n" + "=" * 100)
    print(f"[LABEL] {label}")
    print("=" * 100)

    y_bin = y_df[label].values.astype(int)
    n_pos = int(y_bin.sum())
    n_neg = int(len(y_bin) - n_pos)

    print(f"[INFO] positives={n_pos}, negatives={n_neg}")

    if n_pos < MIN_POSITIVES_PER_LABEL:
        print(f"[SKIP] {label}: too few positives")
        return None

    model = train_xgb_binary(X_df.values, y_bin)

    model_path = os.path.join(MODEL_DIR, f"xgb__{label}.json")
    model.get_booster().save_model(model_path)
    print("[SAVE]", model_path)

    importance = get_xgb_gain_importance(model, feature_names)

    imp_df = pd.DataFrame({
        "feature": feature_names,
        "gain_importance": importance,
        "matched_groups": [";".join(assign_groups(f)) for f in feature_names],
        "interpretable_for_interaction": [is_interpretable_feature(f) for f in feature_names]
    }).sort_values("gain_importance", ascending=False)

    imp_path = os.path.join(TABLE_DIR, f"XGB_gain_importance__{label}.csv")
    imp_df.to_csv(imp_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", imp_path)

    selected_pairs = screen_candidate_pairs(label, model, X_df, feature_names, importance)

    if selected_pairs.empty:
        print(f"[NO USEFUL PAIR] {label}: no pair passed usefulness criteria")
        return pd.DataFrame()

    final_metrics = []

    for _, row in selected_pairs.iterrows():
        pair_id = int(row["plot_pair_id"])
        metrics = plot_final_pair(
            model=model,
            X_df=X_df,
            label=label,
            pair_id=pair_id,
            pair_row=row
        )

        if metrics is not None:
            final_metrics.append(metrics)

    if final_metrics:
        final_df = pd.DataFrame(final_metrics)

        out_path = os.path.join(TABLE_DIR, f"FINAL_plotted_pair_metrics__{label}.csv")
        final_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("[SAVE]", out_path)

        return final_df

    return pd.DataFrame()


# ============================================================
# 12. Main
# ============================================================

def main():
    print("[INFO] Reading feature file:")
    print(FEATURE_FILE)

    df = pd.read_excel(FEATURE_FILE, sheet_name=SHEET_NAME)

    X_df, y_df, smiles_col = build_X_y(df)
    feature_names = X_df.columns.tolist()

    print("[INFO] X shape:", X_df.shape)
    print("[INFO] y shape:", y_df.shape)
    print("[INFO] n_features:", len(feature_names))

    if smiles_col is not None:
        print("[INFO] SMILES column:", smiles_col)

    export_detected_group_features(feature_names)

    meta = {
        "feature_file": FEATURE_FILE,
        "method": "Representative malodor interaction analysis using ICE + 2D PDP + PDP residual",
        "labels": MALODOR_LABELS,
        "selection_policy": [
            "Only malodor descriptors are analyzed.",
            "Only interpretable structure features are used for interaction plots.",
            "Morgan bits are excluded from interaction visualization.",
            "Candidate pairs are selected from chemically meaningful structure groups.",
            "Pairs with insufficient support, redundancy, weak PDP variation, or weak residual signal are not plotted."
        ],
        "interaction_residual_formula": "I(A,B)=f(A,B)-f(A)-f(B)+baseline",
        "plot_policy": {
            "discrete_or_binary_features": "2D block heatmap",
            "continuous_features": "2D contour plot",
            "ICE": "Complementary conditional response visualization"
        },
        "thresholds": {
            "MIN_PDP_RANGE": MIN_PDP_RANGE,
            "MIN_MEAN_ABS_RESIDUAL": MIN_MEAN_ABS_RESIDUAL,
            "MIN_MAX_ABS_RESIDUAL": MIN_MAX_ABS_RESIDUAL,
            "MIN_BINARY_INTERACTION_CONTRAST": MIN_BINARY_INTERACTION_CONTRAST,
            "MAX_BINARY_PHI_CORR": MAX_BINARY_PHI_CORR,
            "MAX_BINARY_JACCARD": MAX_BINARY_JACCARD
        }
    }

    with open(os.path.join(OUT_DIR, "analysis_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    all_final = []

    for label in MALODOR_LABELS:
        if label not in y_df.columns:
            print(f"[WARN] label not found: {label}")
            continue

        result = analyze_one_label(label, X_df, y_df, feature_names)

        if result is not None and not result.empty:
            all_final.append(result)

    if all_final:
        all_df = pd.concat(all_final, axis=0, ignore_index=True)

        out_csv = os.path.join(TABLE_DIR, "ALL_FINAL_plotted_pair_metrics.csv")
        out_xlsx = os.path.join(TABLE_DIR, "ALL_FINAL_plotted_pair_metrics.xlsx")

        all_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
        all_df.to_excel(out_xlsx, index=False)

        print("[SAVE]", out_csv)
        print("[SAVE]", out_xlsx)

    print("\n[DONE]")
    print("Plots:", PLOT_DIR)
    print("Tables:", TABLE_DIR)


if __name__ == "__main__":
    main()

[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.
[INFO] Reading feature file:
./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X shape: (3756, 2595)
[INFO] y shape: (3756, 24)
[INFO] n_features: 2595
[INFO] SMILES column: Canonical_SMILES
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/tables/detected_interpretable_group_features.csv

[LABEL] sulfurous
[INFO] positives=398, negatives=3358
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/models/xgb__sulfurous.json
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/tables/XGB_gain_importance__sulfurous.csv
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/tables/ALL_candidate_pair_decision__sulfurous.csv
[NO USEFUL PAIR] sulfurous: no pair passed usefulness criteria

[LABEL] garlic
[INFO] positives=120, negatives=3636
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/models/xgb__garlic.json
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/tables/XGB_gain_importance__garli

In [9]:
# -*- coding: utf-8 -*-
"""
Read and print feature names used in saved PDP plots.

功能：
1. 读取之前保存的 PDP2D_probability_matrix__*.csv；
2. 读取之前保存的 PDP2D_interaction_residual_matrix__*.csv；
3. 解析每张图对应的 x/y 轴特征名；
4. 打印所有绘图特征名；
5. 保存特征名汇总表，方便后续针对性修改标签显示。
"""

import os
import re
import glob
import numpy as np
import pandas as pd


# ============================================================
# 0. 路径配置
# ============================================================

OLD_OUT_DIR = "./ICE_2D_PDP_representative_malodor_interactions"

TABLE_DIR = os.path.join(OLD_OUT_DIR, "tables")

OUT_SUMMARY_DIR = os.path.join(OLD_OUT_DIR, "feature_name_check")

os.makedirs(OUT_SUMMARY_DIR, exist_ok=True)


# ============================================================
# 1. 解析函数
# ============================================================

def parse_feature_and_value(label_text):
    """
    解析保存矩阵时的行名/列名。

    例如：
    'FG: Carboxylic acid(-COOH)=0.0'
    'FG: N=C=S=1'
    'C(=O)[OH] && (NumAliphaticCarbons >= 10)=0.0'

    这里必须使用 rsplit("=", 1)，
    因为特征名本身可能包含 =、>=、==。
    """
    label_text = str(label_text)

    if "=" not in label_text:
        return label_text, np.nan

    feature, value = label_text.rsplit("=", 1)

    try:
        value = float(value)
    except Exception:
        pass

    return feature, value


def parse_label_and_pair_id_from_filename(filename):
    """
    从文件名中解析 label 和 pair_id。

    示例：
    PDP2D_probability_matrix__cheesy__pair01__FG_Carboxylic_acid...csv
    """
    base = os.path.basename(filename)

    base = base.replace("PDP2D_probability_matrix__", "")
    base = base.replace("PDP2D_interaction_residual_matrix__", "")
    base = base.replace(".csv", "")

    parts = base.split("__")

    label = parts[0] if len(parts) > 0 else "unknown"

    pair_id = "unknown"
    for p in parts:
        if re.match(r"pair\d+", p):
            pair_id = p
            break

    return label, pair_id


def load_feature_info_from_matrix(csv_path, plot_type):
    """
    读取单个 PDP 矩阵，解析 x/y 轴特征名与网格。
    """
    df = pd.read_csv(csv_path, index_col=0)

    col_labels = list(df.columns)
    row_labels = list(df.index)

    if len(col_labels) == 0 or len(row_labels) == 0:
        raise ValueError(f"Empty matrix file: {csv_path}")

    feature_a, _ = parse_feature_and_value(col_labels[0])
    feature_b, _ = parse_feature_and_value(row_labels[0])

    grid_a = []
    grid_b = []

    for c in col_labels:
        _, v = parse_feature_and_value(c)
        grid_a.append(v)

    for r in row_labels:
        _, v = parse_feature_and_value(r)
        grid_b.append(v)

    label, pair_id = parse_label_and_pair_id_from_filename(csv_path)

    info = {
        "label": label,
        "pair_id": pair_id,
        "plot_type": plot_type,
        "feature_a_x_axis": feature_a,
        "feature_b_y_axis": feature_b,
        "grid_a_x_values": "; ".join(map(str, grid_a)),
        "grid_b_y_values": "; ".join(map(str, grid_b)),
        "n_grid_a": len(grid_a),
        "n_grid_b": len(grid_b),
        "matrix_shape": f"{df.shape[0]} x {df.shape[1]}",
        "matrix_file": os.path.basename(csv_path),
        "matrix_path": csv_path
    }

    return info


# ============================================================
# 2. 批量读取所有矩阵文件
# ============================================================

def collect_all_plotted_feature_names():
    prob_files = sorted(
        glob.glob(
            os.path.join(
                TABLE_DIR,
                "PDP2D_probability_matrix__*.csv"
            )
        )
    )

    resid_files = sorted(
        glob.glob(
            os.path.join(
                TABLE_DIR,
                "PDP2D_interaction_residual_matrix__*.csv"
            )
        )
    )

    print(f"[INFO] Found probability matrices: {len(prob_files)}")
    print(f"[INFO] Found residual matrices: {len(resid_files)}")

    rows = []

    for csv_path in prob_files:
        try:
            rows.append(
                load_feature_info_from_matrix(
                    csv_path=csv_path,
                    plot_type="probability"
                )
            )
        except Exception as e:
            print(f"[ERROR] Failed to read probability matrix: {csv_path}")
            print(e)

    for csv_path in resid_files:
        try:
            rows.append(
                load_feature_info_from_matrix(
                    csv_path=csv_path,
                    plot_type="interaction_residual"
                )
            )
        except Exception as e:
            print(f"[ERROR] Failed to read residual matrix: {csv_path}")
            print(e)

    summary_df = pd.DataFrame(rows)

    if summary_df.empty:
        print("[WARN] No PDP matrix files found.")
        return summary_df

    summary_df = summary_df.sort_values(
        ["label", "pair_id", "plot_type"]
    ).reset_index(drop=True)

    return summary_df


# ============================================================
# 3. 打印与保存
# ============================================================

def print_feature_summary(summary_df):
    """
    按 label / pair_id 打印绘图特征名。
    """
    if summary_df.empty:
        print("[WARN] summary_df is empty.")
        return

    print("\n" + "=" * 120)
    print("PDP 绘图特征名汇总")
    print("=" * 120)

    grouped = summary_df.groupby(["label", "pair_id"], sort=True)

    for (label, pair_id), g in grouped:
        first = g.iloc[0]

        print(f"\n[LABEL] {label} | [PAIR] {pair_id}")
        print("-" * 100)
        print(f"x-axis feature_a: {first['feature_a_x_axis']}")
        print(f"y-axis feature_b: {first['feature_b_y_axis']}")
        print(f"x grid: {first['grid_a_x_values']}")
        print(f"y grid: {first['grid_b_y_values']}")

        plot_types = ", ".join(g["plot_type"].tolist())
        print(f"plot types: {plot_types}")


def save_feature_summary(summary_df):
    """
    保存绘图特征名汇总表。
    """
    if summary_df.empty:
        return

    out_csv = os.path.join(
        OUT_SUMMARY_DIR,
        "PDP_plotted_feature_name_summary.csv"
    )

    out_xlsx = os.path.join(
        OUT_SUMMARY_DIR,
        "PDP_plotted_feature_name_summary.xlsx"
    )

    summary_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(out_xlsx, index=False)

    print("\n[SAVE]", out_csv)
    print("[SAVE]", out_xlsx)


def save_unique_feature_names(summary_df):
    """
    保存所有唯一出现过的绘图特征名，方便你后续逐个设置短标签。
    """
    if summary_df.empty:
        return

    feature_a = summary_df[["feature_a_x_axis"]].rename(
        columns={"feature_a_x_axis": "feature"}
    )

    feature_a["axis"] = "x"

    feature_b = summary_df[["feature_b_y_axis"]].rename(
        columns={"feature_b_y_axis": "feature"}
    )

    feature_b["axis"] = "y"

    unique_df = pd.concat([feature_a, feature_b], axis=0, ignore_index=True)

    unique_df = (
        unique_df
        .groupby("feature", as_index=False)
        .agg(
            used_as_axis=("axis", lambda x: "; ".join(sorted(set(x)))),
            count=("axis", "count")
        )
        .sort_values(["count", "feature"], ascending=[False, True])
        .reset_index(drop=True)
    )

    out_csv = os.path.join(
        OUT_SUMMARY_DIR,
        "PDP_unique_feature_names_for_label_mapping.csv"
    )

    out_xlsx = os.path.join(
        OUT_SUMMARY_DIR,
        "PDP_unique_feature_names_for_label_mapping.xlsx"
    )

    unique_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    unique_df.to_excel(out_xlsx, index=False)

    print("[SAVE]", out_csv)
    print("[SAVE]", out_xlsx)

    print("\n" + "=" * 120)
    print("唯一绘图特征名列表")
    print("=" * 120)

    for i, row in unique_df.iterrows():
        print(f"{i + 1:02d}. [{row['used_as_axis']}] {row['feature']}")


# ============================================================
# 4. 主程序
# ============================================================

def main():
    summary_df = collect_all_plotted_feature_names()

    print_feature_summary(summary_df)

    save_feature_summary(summary_df)

    save_unique_feature_names(summary_df)

    print("\n[DONE] Feature name checking finished.")
    print("Output folder:", OUT_SUMMARY_DIR)


if __name__ == "__main__":
    main()

[INFO] Found probability matrices: 10
[INFO] Found residual matrices: 10

PDP 绘图特征名汇总

[LABEL] cheesy | [PAIR] pair01
----------------------------------------------------------------------------------------------------
x-axis feature_a: FG: Carboxylic acid(–COOH)
y-axis feature_b: C(=O)O&&(TPSA>60)&&(LogP<1.0)
x grid: 0.0; 1.0
y grid: 0.0; 1.0
plot types: interaction_residual, probability

[LABEL] cheesy | [PAIR] pair02
----------------------------------------------------------------------------------------------------
x-axis feature_a: FG: Carboxylic acid(–COOH)
y-axis feature_b: KG_Ancestor__GroupsContainingSulfur
x grid: 0.0; 1.0
y grid: 0.0; 1.0
plot types: interaction_residual, probability

[LABEL] cheesy | [PAIR] pair03
----------------------------------------------------------------------------------------------------
x-axis feature_a: FG: Carboxylic acid(–COOH)
y-axis feature_b: FG: [CX3](=O)[#6][#6]
x grid: 0.0; 1.0
y grid: 0.0; 1.0
plot types: interaction_residual, probabilit

In [12]:
# -*- coding: utf-8 -*-
"""
Redraw saved 2D PDP and PDP interaction residual figures with customized feature labels.

功能：
1. 读取之前保存的：
   - PDP2D_probability_matrix__*.csv
   - PDP2D_interaction_residual_matrix__*.csv

2. 不重新训练模型；
3. 不重新计算 PDP；
4. 只根据已保存矩阵重新绘图；
5. 去掉图标题；
6. 统一蓝白色系；
7. 使用 Arial 字体，如果环境没有 Arial，则自动使用 DejaVu Sans；
8. 根据实际打印出的特征名进行精确标签替换；
9. 解决 y 轴标签过长、竖向挤压的问题；
10. 图片只保存 PNG；
11. 图片 dpi = 800。
"""

import os
import re
import glob
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import font_manager as fm


# ============================================================
# 0. 路径配置
# ============================================================

OLD_OUT_DIR = "./ICE_2D_PDP_representative_malodor_interactions"

TABLE_DIR = os.path.join(OLD_OUT_DIR, "tables")

# 新图保存路径，避免覆盖原图
NEW_PLOT_DIR = os.path.join(
    OLD_OUT_DIR,
    "plots_redraw_no_title_blue_white_custom_labels_dpi800"
)

os.makedirs(NEW_PLOT_DIR, exist_ok=True)

# 图片保存 dpi
DPI = 800


# ============================================================
# 1. 字体、字号、颜色配置
# ============================================================

# x/y 轴标签字体大小
FONT_X_LABEL = 16
FONT_Y_LABEL = 16

# 坐标轴刻度字体大小
FONT_TICK = 18

# block heatmap 内部数字字体大小
FONT_ANNOT = 20

# probability 色条标题字体大小
CBAR_LABEL_FONT_PROB = 14

# residual 色条标题字体大小
CBAR_LABEL_FONT_RESID = 18

# 色条刻度字体大小
CBAR_TICK_FONT = 18

# 坐标轴线宽和刻度宽度
AXIS_LINEWIDTH = 2.2
TICK_WIDTH = 2.0
TICK_LENGTH = 6

# 离散变量判断阈值
MAX_UNIQUE_AS_DISCRETE = 6

# residual 图是否用 abs(residual) 着色
# True：颜色表示交互强度大小，图中数字仍保留正负号
# False：颜色直接表示 signed residual，但蓝白色不适合区分正负
RESIDUAL_COLOR_BY_ABS = True

# residual 色条标题是否显示绝对值符号
# False：显示 "PDP interaction residual"
# True：显示 "|PDP interaction residual|"
ADD_ABS_SYMBOL_TO_RESIDUAL_CBAR = False

# 只保存 PNG，不保存 PDF/SVG
SAVE_PDF = False
SAVE_SVG = False


# 蓝白色系
BLUE_WHITE_CMAP = LinearSegmentedColormap.from_list(
    "custom_blue_white",
    [
        "#ffffff",
        "#eff6fb",
        "#d9eaf7",
        "#bdd7e7",
        "#6baed6",
        "#3182bd",
        "#08519c",
        "#08306b"
    ]
)


# ============================================================
# 2. 特征名精确映射
# ============================================================

FEATURE_LABEL_MAP = {
    # x 轴常见特征
    "FG: Carboxylic acid(–COOH)": "Carboxylic acid (-COOH)",
    "FG: Carboxylic acid(-COOH)": "Carboxylic acid (-COOH)",

    "KG_Ancestor__GroupsContainingNitrogen": "N-containing groups",
    "KG_Ancestor__GroupsContainingSulfur": "S-containing groups",
    "KG_Ancestor__Amines": "Amines",

    # y 轴特征
    "KG_FG__Ether": "Ether",
    "KG_Ancestor__Ether": "Ether-related groups",

    "Atom count: N >= 2": "N atom count",

    "C(=O)O&&(TPSA>60)&&(LogP<1.0)": "C(=O)O && TPSA>60 && LogP<1.0",
    "C(=O)O && (TPSA>60) && (LogP<1.0)": "C(=O)O && TPSA>60 && LogP<1.0",
    "C(=O)O && (TPSA > 60) && (LogP < 1.0)": "C(=O)O && TPSA>60 && LogP<1.0",

    "C(=O)[OH] && (MolWt < 110)": "C(=O)[OH] && MolWt<110",
    "C(=O)[OH]&&(MolWt<110)": "C(=O)[OH] && MolWt<110",

    "C(=O)[OH] && (NumAliphaticCarbons >= 10)": "C(=O)[OH] && Aliphatic C≥10",
    "C(=O)[OH]&&(NumAliphaticCarbons>=10)": "C(=O)[OH] && Aliphatic C≥10",

    "FG: CC(=O)O": "CC(=O)O",
    "FG: [CX3](=O)[#6][#6]": "Ketone-like carbonyl",
}


def normalize_feature_key(s):
    """
    归一化特征名，增强匹配鲁棒性。
    """
    s = str(s)
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = s.replace("≥", ">=").replace("≤", "<=")
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "", s)
    return s.lower()


FEATURE_LABEL_MAP_NORM = {
    normalize_feature_key(k): v
    for k, v in FEATURE_LABEL_MAP.items()
}


def pretty_feature_label(feature_name, axis="x"):
    """
    根据 FEATURE_LABEL_MAP 返回适合绘图的短标签。
    如果没有命中映射，则进行保守自动简化。
    """
    raw = str(feature_name)

    # 1. 精确匹配
    if raw in FEATURE_LABEL_MAP:
        return FEATURE_LABEL_MAP[raw]

    # 2. 归一化匹配
    norm = normalize_feature_key(raw)
    if norm in FEATURE_LABEL_MAP_NORM:
        return FEATURE_LABEL_MAP_NORM[norm]

    # 3. 没有映射时的保守处理
    s = raw
    s = s.replace("KG_Ancestor__", "")
    s = s.replace("KG_FG__", "")
    s = s.replace("FG: ", "")
    s = s.replace("GroupsContainingSulfur", "S-containing groups")
    s = s.replace("GroupsContainingNitrogen", "N-containing groups")
    s = s.replace("NumAliphaticCarbons", "Aliphatic C")
    s = s.replace("MolWt", "MW")
    s = s.replace("&&", "\n")
    s = s.replace(">=", "≥")
    s = s.replace("<=", "≤")

    width = 22 if axis == "y" else 30

    if len(s) > width:
        s = "\n".join(
            textwrap.wrap(
                s,
                width=width,
                break_long_words=False,
                break_on_hyphens=False
            )
        )

    return s


# ============================================================
# 3. Matplotlib 全局样式
# ============================================================

def setup_matplotlib_style():
    available_fonts = {f.name for f in fm.fontManager.ttflist}

    if "Arial" in available_fonts:
        font_family = "Arial"
    else:
        font_family = "DejaVu Sans"
        print("[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.")

    mpl.rcParams.update({
        "font.family": font_family,
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "xtick.labelsize": FONT_TICK,
        "ytick.labelsize": FONT_TICK,
        "axes.linewidth": AXIS_LINEWIDTH,
        "xtick.major.width": TICK_WIDTH,
        "ytick.major.width": TICK_WIDTH,
        "xtick.major.size": TICK_LENGTH,
        "ytick.major.size": TICK_LENGTH,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.unicode_minus": False,
    })


setup_matplotlib_style()


def style_axis(ax):
    ax.tick_params(
        axis="both",
        which="major",
        width=TICK_WIDTH,
        length=TICK_LENGTH,
        labelsize=FONT_TICK
    )

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(AXIS_LINEWIDTH)


def style_colorbar(cbar, label, label_fontsize):
    cbar.set_label(
        label,
        fontsize=label_fontsize,
        fontweight="bold",
        labelpad=12
    )

    cbar.ax.tick_params(
        labelsize=CBAR_TICK_FONT,
        width=TICK_WIDTH,
        length=TICK_LENGTH
    )

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")


# ============================================================
# 4. 文件名和矩阵解析
# ============================================================

def sanitize_filename(s):
    return re.sub(r"[^\w\-_\.]+", "_", str(s))


def parse_feature_and_value(label_text):
    """
    解析保存矩阵时的行名/列名。

    注意：
    使用 rsplit("=", 1)，避免特征名中本身包含 =、>=、== 时解析错误。
    """
    label_text = str(label_text)

    if "=" not in label_text:
        return label_text, np.nan

    feature, value = label_text.rsplit("=", 1)

    try:
        value = float(value)
    except Exception:
        pass

    return feature, value


def load_pdp_matrix(csv_path):
    """
    读取 PDP 矩阵文件。

    返回：
    matrix, grid_a, grid_b, feature_a, feature_b
    """
    df = pd.read_csv(csv_path, index_col=0)

    col_labels = list(df.columns)
    row_labels = list(df.index)

    if len(col_labels) == 0 or len(row_labels) == 0:
        raise ValueError(f"Empty matrix file: {csv_path}")

    feature_a, _ = parse_feature_and_value(col_labels[0])
    feature_b, _ = parse_feature_and_value(row_labels[0])

    grid_a = []
    grid_b = []

    for c in col_labels:
        _, v = parse_feature_and_value(c)
        grid_a.append(v)

    for r in row_labels:
        _, v = parse_feature_and_value(r)
        grid_b.append(v)

    grid_a = np.array(grid_a, dtype=float)
    grid_b = np.array(grid_b, dtype=float)

    matrix = df.values.astype(float)

    return matrix, grid_a, grid_b, feature_a, feature_b


def is_discrete_grid(grid):
    return len(grid) <= MAX_UNIQUE_AS_DISCRETE


def format_grid_labels(grid):
    labels = []

    for v in grid:
        try:
            vf = float(v)
            if vf.is_integer():
                labels.append(str(int(vf)))
            else:
                labels.append(f"{vf:.3g}")
        except Exception:
            labels.append(str(v))

    return labels


# ============================================================
# 5. 保存图像
# ============================================================

def save_figure(fig, out_png):
    """
    只保存 PNG，不保存 PDF/SVG。
    """
    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    print("[SAVE]", out_png)


# ============================================================
# 6. 绘图：离散 / 二值 block heatmap
# ============================================================

def plot_discrete_heatmap_from_matrix(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    cbar_label,
    out_png,
    signed=False,
    cbar_label_fontsize=22
):
    mat = np.asarray(matrix, dtype=float)

    if signed and RESIDUAL_COLOR_BY_ABS:
        color_mat = np.abs(mat)
        vmin = 0.0
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0

        if ADD_ABS_SYMBOL_TO_RESIDUAL_CBAR:
            colorbar_label = f"|{cbar_label}|"
        else:
            colorbar_label = cbar_label

    else:
        color_mat = mat
        vmin = float(np.nanmin(color_mat))
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0

        colorbar_label = cbar_label

    # 对 2x2 或 4x2 图设置更稳定的画布比例
    fig_w = max(6.6, 1.55 * len(grid_a) + 3.6)
    fig_h = max(5.4, 1.15 * len(grid_b) + 2.8)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        color_mat,
        origin="lower",
        aspect="auto",
        cmap=BLUE_WHITE_CMAP,
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(im, ax=ax)
    style_colorbar(
        cbar=cbar,
        label=colorbar_label,
        label_fontsize=cbar_label_fontsize
    )

    ax.set_xticks(np.arange(len(grid_a)))
    ax.set_yticks(np.arange(len(grid_b)))

    ax.set_xticklabels(
        format_grid_labels(grid_a),
        fontweight="bold"
    )

    ax.set_yticklabels(
        format_grid_labels(grid_b),
        fontweight="bold"
    )

    ax.set_xlabel(
        pretty_feature_label(feature_a, axis="x"),
        fontsize=FONT_X_LABEL,
        fontweight="bold",
        labelpad=10
    )

    ax.set_ylabel(
        pretty_feature_label(feature_b, axis="y"),
        fontsize=FONT_Y_LABEL,
        fontweight="bold",
        labelpad=12
    )

    # 去掉标题
    ax.set_title("")

    # 数值标注
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            raw_val = mat[i, j]
            color_val = color_mat[i, j]

            if not np.isfinite(raw_val):
                continue

            if signed:
                text = f"{raw_val:+.3f}"
            else:
                text = f"{raw_val:.3f}"

            if vmax > vmin:
                threshold = vmin + 0.62 * (vmax - vmin)
                text_color = "white" if color_val > threshold else "black"
            else:
                text_color = "black"

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=FONT_ANNOT,
                fontweight="bold",
                color=text_color
            )

    # 白色网格线强调离散组合
    ax.set_xticks(np.arange(-0.5, len(grid_a), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(grid_b), 1), minor=True)
    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=2.0
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    style_axis(ax)

    # 留出左侧和右侧空间，避免 y 标签和 colorbar 被挤压
    plt.tight_layout()
    plt.subplots_adjust(left=0.20, right=0.88)

    save_figure(fig, out_png)
    plt.close(fig)


# ============================================================
# 7. 绘图：连续 contour plot
# ============================================================

def plot_contour_from_matrix(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    cbar_label,
    out_png,
    signed=False,
    cbar_label_fontsize=22
):
    mat = np.asarray(matrix, dtype=float)
    Xg, Yg = np.meshgrid(grid_a, grid_b)

    if signed and RESIDUAL_COLOR_BY_ABS:
        color_mat = np.abs(mat)
        vmin = 0.0
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0

        levels = np.linspace(vmin, vmax, 24)

        if ADD_ABS_SYMBOL_TO_RESIDUAL_CBAR:
            colorbar_label = f"|{cbar_label}|"
        else:
            colorbar_label = cbar_label

    else:
        color_mat = mat
        vmin = float(np.nanmin(color_mat))
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0

        levels = np.linspace(vmin, vmax, 24)
        colorbar_label = cbar_label

    fig, ax = plt.subplots(figsize=(7.8, 6.3))

    cf = ax.contourf(
        Xg,
        Yg,
        color_mat,
        levels=levels,
        cmap=BLUE_WHITE_CMAP,
        extend="both"
    )

    cbar = plt.colorbar(cf, ax=ax)
    style_colorbar(
        cbar=cbar,
        label=colorbar_label,
        label_fontsize=cbar_label_fontsize
    )

    # 等高线标注原始 matrix 数值
    try:
        cs = ax.contour(
            Xg,
            Yg,
            mat,
            levels=8,
            colors="#08306b",
            linewidths=1.3,
            alpha=0.85
        )

        ax.clabel(
            cs,
            inline=True,
            fontsize=FONT_ANNOT - 4,
            fmt="%.3f",
            colors="#08306b"
        )

    except Exception:
        pass

    ax.set_xlabel(
        pretty_feature_label(feature_a, axis="x"),
        fontsize=FONT_X_LABEL,
        fontweight="bold",
        labelpad=10
    )

    ax.set_ylabel(
        pretty_feature_label(feature_b, axis="y"),
        fontsize=FONT_Y_LABEL,
        fontweight="bold",
        labelpad=12
    )

    # 去掉标题
    ax.set_title("")

    style_axis(ax)

    plt.tight_layout()
    plt.subplots_adjust(left=0.22, right=0.88)

    save_figure(fig, out_png)
    plt.close(fig)


# ============================================================
# 8. 自动判断离散 / 连续并绘图
# ============================================================

def plot_pdp_matrix_auto(
    csv_path,
    out_png,
    cbar_label,
    signed=False,
    cbar_label_fontsize=22
):
    matrix, grid_a, grid_b, feature_a, feature_b = load_pdp_matrix(csv_path)

    print("\n[PLOT]")
    print("file:", os.path.basename(csv_path))
    print("x-axis raw:", feature_a)
    print("x-axis display:", pretty_feature_label(feature_a, axis="x").replace("\n", " / "))
    print("y-axis raw:", feature_b)
    print("y-axis display:", pretty_feature_label(feature_b, axis="y").replace("\n", " / "))
    print("x grid:", grid_a)
    print("y grid:", grid_b)

    if is_discrete_grid(grid_a) or is_discrete_grid(grid_b):
        plot_discrete_heatmap_from_matrix(
            matrix=matrix,
            grid_a=grid_a,
            grid_b=grid_b,
            feature_a=feature_a,
            feature_b=feature_b,
            cbar_label=cbar_label,
            out_png=out_png,
            signed=signed,
            cbar_label_fontsize=cbar_label_fontsize
        )

    else:
        plot_contour_from_matrix(
            matrix=matrix,
            grid_a=grid_a,
            grid_b=grid_b,
            feature_a=feature_a,
            feature_b=feature_b,
            cbar_label=cbar_label,
            out_png=out_png,
            signed=signed,
            cbar_label_fontsize=cbar_label_fontsize
        )


# ============================================================
# 9. 批量重画所有已保存 PDP 图
# ============================================================

def redraw_all_saved_pdp_figures():
    prob_files = sorted(
        glob.glob(
            os.path.join(
                TABLE_DIR,
                "PDP2D_probability_matrix__*.csv"
            )
        )
    )

    resid_files = sorted(
        glob.glob(
            os.path.join(
                TABLE_DIR,
                "PDP2D_interaction_residual_matrix__*.csv"
            )
        )
    )

    print(f"[INFO] Found probability matrices: {len(prob_files)}")
    print(f"[INFO] Found residual matrices: {len(resid_files)}")

    # 1. PDP probability
    for csv_path in prob_files:
        base = os.path.basename(csv_path)
        name = base.replace(
            "PDP2D_probability_matrix__",
            ""
        ).replace(
            ".csv",
            ""
        )

        out_png = os.path.join(
            NEW_PLOT_DIR,
            f"REDRAW_PDP2D_probability__{name}.png"
        )

        plot_pdp_matrix_auto(
            csv_path=csv_path,
            out_png=out_png,
            cbar_label="Partial dependence/predicted probability",
            signed=False,
            cbar_label_fontsize=CBAR_LABEL_FONT_PROB
        )

    # 2. PDP interaction residual
    for csv_path in resid_files:
        base = os.path.basename(csv_path)
        name = base.replace(
            "PDP2D_interaction_residual_matrix__",
            ""
        ).replace(
            ".csv",
            ""
        )

        out_png = os.path.join(
            NEW_PLOT_DIR,
            f"REDRAW_PDP2D_interaction_residual__{name}.png"
        )

        plot_pdp_matrix_auto(
            csv_path=csv_path,
            out_png=out_png,
            cbar_label="PDP interaction residual",
            signed=True,
            cbar_label_fontsize=CBAR_LABEL_FONT_RESID
        )

    print("\n[DONE] Redraw finished.")
    print("New plots:", NEW_PLOT_DIR)


# ============================================================
# 10. 只重画指定 label
# ============================================================

def redraw_one_label(label):
    prob_files = sorted(
        glob.glob(
            os.path.join(
                TABLE_DIR,
                f"PDP2D_probability_matrix__{label}__*.csv"
            )
        )
    )

    resid_files = sorted(
        glob.glob(
            os.path.join(
                TABLE_DIR,
                f"PDP2D_interaction_residual_matrix__{label}__*.csv"
            )
        )
    )

    print(f"[INFO] Label = {label}")
    print(f"[INFO] Found probability matrices: {len(prob_files)}")
    print(f"[INFO] Found residual matrices: {len(resid_files)}")

    for csv_path in prob_files:
        base = os.path.basename(csv_path)
        name = base.replace(
            "PDP2D_probability_matrix__",
            ""
        ).replace(
            ".csv",
            ""
        )

        out_png = os.path.join(
            NEW_PLOT_DIR,
            f"REDRAW_PDP2D_probability__{name}.png"
        )

        plot_pdp_matrix_auto(
            csv_path=csv_path,
            out_png=out_png,
            cbar_label="Partial dependence/predicted probability",
            signed=False,
            cbar_label_fontsize=CBAR_LABEL_FONT_PROB
        )

    for csv_path in resid_files:
        base = os.path.basename(csv_path)
        name = base.replace(
            "PDP2D_interaction_residual_matrix__",
            ""
        ).replace(
            ".csv",
            ""
        )

        out_png = os.path.join(
            NEW_PLOT_DIR,
            f"REDRAW_PDP2D_interaction_residual__{name}.png"
        )

        plot_pdp_matrix_auto(
            csv_path=csv_path,
            out_png=out_png,
            cbar_label="PDP interaction residual",
            signed=True,
            cbar_label_fontsize=CBAR_LABEL_FONT_RESID
        )

    print("\n[DONE] Redraw one label finished.")
    print("New plots:", NEW_PLOT_DIR)


# ============================================================
# 11. 运行
# ============================================================

if __name__ == "__main__":
    redraw_all_saved_pdp_figures()

    # 只想重画某一个标签时，注释上一行，取消下面任意一行注释：
    # redraw_one_label("cheesy")
    # redraw_one_label("fishy")
    # redraw_one_label("garlic")
    # redraw_one_label("sour")

[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.
[INFO] Found probability matrices: 10
[INFO] Found residual matrices: 10

[PLOT]
file: PDP2D_probability_matrix__cheesy__pair01__FG_Carboxylic_acid_COOH___x__C_O_O_TPSA_60_LogP_1.0_.csv
x-axis raw: FG: Carboxylic acid(–COOH)
x-axis display: Carboxylic acid (-COOH)
y-axis raw: C(=O)O&&(TPSA>60)&&(LogP<1.0)
y-axis display: C(=O)O && TPSA>60 && LogP<1.0
x grid: [0. 1.]
y grid: [0. 1.]
[SAVE] ./ICE_2D_PDP_representative_malodor_interactions/plots_redraw_no_title_blue_white_custom_labels_dpi800/REDRAW_PDP2D_probability__cheesy__pair01__FG_Carboxylic_acid_COOH___x__C_O_O_TPSA_60_LogP_1.0_.png

[PLOT]
file: PDP2D_probability_matrix__cheesy__pair02__FG_Carboxylic_acid_COOH___x__KG_Ancestor__GroupsContainingSulfur.csv
x-axis raw: FG: Carboxylic acid(–COOH)
x-axis display: Carboxylic acid (-COOH)
y-axis raw: KG_Ancestor__GroupsContainingSulfur
y-axis display: S-containing groups
x grid: [0. 1.]
y grid: [0. 1.]
[SAVE] ./ICE_2D_